## 8-Queens Problem as a Constraint Satisfaction Problem (CSP)

Algorithms:
1. Plain Backtracking
2. Backtracking + MRV + LCV
3. Backtracking + MRV + LCV + AC-3

## CSP Representation

The 8-Queens problem is represented as a CSP.

- Variables: Q1, Q2, ..., Q8
- Domain: {1, 2, 3, 4, 5, 6, 7, 8}
- Qi represents the row selected for the queen in column i.

Constraints:
- No two queens can be in the same row.
- No two queens can be on the same diagonal.

For two queens in columns i and j:

|Qi - Qj| != |i - j|

In [40]:
N = 8

variables = list(range(1, N + 1))
values = list(range(1, N + 1))

print("Variables:", ["Q" + str(i) for i in variables])
print("Domain:", values)

Variables: ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8']
Domain: [1, 2, 3, 4, 5, 6, 7, 8]


## Constraint Checking

A value can be assigned to a queen only when it does not conflict with
any queen that has already been assigned.

Two queens conflict when:
- they are in the same row, or
- they are on the same diagonal.

In [41]:
def is_consistent(var, value, assignment):
    for other_var, other_value in assignment.items():

        # Same row
        if value == other_value:
            return False

        # Same diagonal
        if abs(value - other_value) == abs(var - other_var):
            return False

    return True

In [42]:
def print_board(solution):
    for row in range(1, N + 1):
        for col in range(1, N + 1):

            if solution[col] == row:
                print("Q", end=" ")
            else:
                print(".", end=" ")

        print()

## 1. Plain Backtracking

Backtracking assigns one variable at a time.

The variables are selected in the order Q1, Q2, ..., Q8.

For every variable, each possible row is tried.
If the value is consistent, the queen is assigned and the next variable is considered.

If the branch fails, the assignment is removed and another value is tried.

For this lab, a search step is counted whenever a row value is attempted.

In [43]:
def plain_backtracking():
    assignment = {}
    steps = 0

    def backtrack(col):
        nonlocal steps

        if col > N:
            return assignment.copy()

        for row in values:

            steps += 1

            if is_consistent(col, row, assignment):
                assignment[col] = row

                result = backtrack(col + 1)

                if result is not None:
                    return result

                del assignment[col]

        return None

    solution = backtrack(1)

    return solution, steps

In [44]:
solution1, steps1 = plain_backtracking()

print("Solution:")
print([solution1[i] for i in variables])

print("\nChessboard:")
print_board(solution1)

print("\nSearch Steps:", steps1)

Solution:
[1, 5, 8, 6, 3, 7, 2, 4]

Chessboard:
Q . . . . . . . 
. . . . . . Q . 
. . . . Q . . . 
. . . . . . . Q 
. Q . . . . . . 
. . . Q . . . . 
. . . . . Q . . 
. . Q . . . . . 

Search Steps: 876


## 2. MRV — Minimum Remaining Values

MRV selects the unassigned variable having the fewest legal values.

It helps us choose the variable that is currently the most restricted.

MRV answers:

Which variable should be assigned next?

In [45]:
def get_domains(assignment):
    domains = {}

    for var in variables:

        if var in assignment:
            continue

        legal = []

        for value in values:
            if is_consistent(var, value, assignment):
                legal.append(value)

        domains[var] = legal

    return domains


def select_mrv(assignment):
    domains = get_domains(assignment)

    if not domains:
        return None, domains

    var = min(
        domains,
        key=lambda x: (len(domains[x]), x)
    )

    return var, domains

## LCV — Least Constraining Value

After selecting a variable using MRV, LCV decides the order in which
its values should be tried.

A value is less constraining if it eliminates fewer legal choices
from the other unassigned variables.

LCV answers:

Which value should be tried first?

In [46]:
def order_lcv(var, domain, assignment):
    scores = []

    for value in domain:
        eliminated = 0

        for other_var in variables:

            if other_var == var or other_var in assignment:
                continue

            for other_value in values:

                if not is_consistent(
                    other_var,
                    other_value,
                    assignment
                ):
                    continue

                if (
                    value == other_value
                    or abs(value - other_value)
                    == abs(var - other_var)
                ):
                    eliminated += 1

        scores.append((eliminated, value))

    scores.sort()

    return [value for eliminated, value in scores]

In [47]:
def mrv_lcv_backtracking():
    assignment = {}
    steps = 0

    def backtrack():
        nonlocal steps

        if len(assignment) == N:
            return assignment.copy()

        var, domains = select_mrv(assignment)

        if not domains[var]:
            return None

        print("\nMRV selected variable: Q" + str(var))
        print("Domain:", domains[var])

        ordered_values = order_lcv(
            var,
            domains[var],
            assignment
        )

        print(
            "LCV order for Q{}: {}".format(
                var,
                ordered_values
            )
        )

        for value in ordered_values:

            steps += 1

            if is_consistent(var, value, assignment):
                assignment[var] = value

                result = backtrack()

                if result is not None:
                    return result

                del assignment[var]

        return None

    solution = backtrack()

    return solution, steps

In [48]:
solution2, steps2 = mrv_lcv_backtracking()

print("\nSolution:")
print([solution2[i] for i in variables])

print("\nChessboard:")
print_board(solution2)

print("\nSearch Steps:", steps2)


MRV selected variable: Q1
Domain: [1, 2, 3, 4, 5, 6, 7, 8]
LCV order for Q1: [1, 2, 3, 4, 5, 6, 7, 8]

MRV selected variable: Q2
Domain: [3, 4, 5, 6, 7, 8]
LCV order for Q2: [4, 6, 8, 3, 5, 7]

MRV selected variable: Q3
Domain: [2, 6, 7, 8]
LCV order for Q3: [7, 8, 2, 6]

MRV selected variable: Q4
Domain: [3, 5]
LCV order for Q4: [3, 5]

MRV selected variable: Q6
Domain: [2]
LCV order for Q6: [2]

MRV selected variable: Q5
Domain: [6, 8]
LCV order for Q5: [6, 8]

MRV selected variable: Q7
Domain: [5]
LCV order for Q7: [5]

MRV selected variable: Q7
Domain: [5]
LCV order for Q7: [5]

MRV selected variable: Q6
Domain: [2]
LCV order for Q6: [2]

MRV selected variable: Q5
Domain: [8]
LCV order for Q5: [8]

MRV selected variable: Q4
Domain: [3, 5]
LCV order for Q4: [5, 3]

MRV selected variable: Q6
Domain: [2]
LCV order for Q6: [2]

MRV selected variable: Q6
Domain: [3, 7]
LCV order for Q6: [3, 7]

MRV selected variable: Q8
Domain: [6]
LCV order for Q8: [6]

MRV selected variable: Q5
Domai

## 3. AC-3 — Arc Consistency

For an arc Xi -> Xj, every value in Xi must have at least one
compatible value in Xj.

If a value has no supporting value in the other domain, it is removed.

When a domain changes, the related arcs are checked again.

If a domain becomes empty, the current branch fails.

In [49]:
from collections import deque


def revise(domains, xi, xj):
    revised = False

    for x in domains[xi].copy():

        supported = False

        for y in domains[xj]:

            if (
                x != y
                and abs(x - y) != abs(xi - xj)
            ):
                supported = True
                break

        if not supported:
            domains[xi].remove(x)
            revised = True

    return revised

In [50]:
def ac3(domains):
    queue = deque()

    for xi in variables:
        for xj in variables:

            if xi != xj:
                queue.append((xi, xj))

    while queue:

        xi, xj = queue.popleft()

        old_domain = domains[xi].copy()

        if revise(domains, xi, xj):

            print(
                "AC-3: Q{} <- Q{} : {} -> {}".format(
                    xi,
                    xj,
                    old_domain,
                    domains[xi]
                )
            )

            if not domains[xi]:
                return False

            for xk in variables:

                if xk != xi and xk != xj:
                    queue.append((xk, xi))

    return True

## MRV + LCV + AC-3

For this version:

1. MRV selects the variable.
2. LCV gives the order of values.
3. A value is assigned.
4. AC-3 is applied to the domains.
5. If AC-3 produces an empty domain, the value is rejected.
6. Otherwise the search continues.
7. If the branch fails later, backtracking is performed.

In [51]:
def mrv_lcv_ac3_backtracking():
    assignment = {}

    domains = {
        var: values.copy()
        for var in variables
    }

    steps = 0

    def backtrack(current_domains):
        nonlocal steps

        if len(assignment) == N:
            return assignment.copy()

        unassigned = [
            var for var in variables
            if var not in assignment
        ]

        var = min(
            unassigned,
            key=lambda x: (len(current_domains[x]), x)
        )

        print("\nMRV selected variable: Q" + str(var))
        print("Domain:", current_domains[var])

        ordered_values = order_lcv(
            var,
            current_domains[var],
            assignment
        )

        print(
            "LCV order for Q{}: {}".format(
                var,
                ordered_values
            )
        )

        for value in ordered_values:

            steps += 1

            if value not in current_domains[var]:
                continue

            if not is_consistent(
                var,
                value,
                assignment
            ):
                continue

            new_domains = {
                v: current_domains[v].copy()
                for v in variables
            }

            new_domains[var] = [value]

            assignment[var] = value

            print(
                "Trying Q{} = {}".format(
                    var,
                    value
                )
            )

            if ac3(new_domains):

                result = backtrack(new_domains)

                if result is not None:
                    return result

            else:
                print(
                    "AC-3 detected failure for Q{} = {}".format(
                        var,
                        value
                    )
                )

            del assignment[var]

        return None

    solution = backtrack(domains)

    return solution, steps

In [52]:
solution3, steps3 = mrv_lcv_ac3_backtracking()

print("\nSolution:")
print([solution3[i] for i in variables])

print("\nChessboard:")
print_board(solution3)

print("\nSearch Steps:", steps3)


MRV selected variable: Q1
Domain: [1, 2, 3, 4, 5, 6, 7, 8]
LCV order for Q1: [1, 2, 3, 4, 5, 6, 7, 8]
Trying Q1 = 1
AC-3: Q2 <- Q1 : [1, 2, 3, 4, 5, 6, 7, 8] -> [3, 4, 5, 6, 7, 8]
AC-3: Q3 <- Q1 : [1, 2, 3, 4, 5, 6, 7, 8] -> [2, 4, 5, 6, 7, 8]
AC-3: Q4 <- Q1 : [1, 2, 3, 4, 5, 6, 7, 8] -> [2, 3, 5, 6, 7, 8]
AC-3: Q5 <- Q1 : [1, 2, 3, 4, 5, 6, 7, 8] -> [2, 3, 4, 6, 7, 8]
AC-3: Q6 <- Q1 : [1, 2, 3, 4, 5, 6, 7, 8] -> [2, 3, 4, 5, 7, 8]
AC-3: Q7 <- Q1 : [1, 2, 3, 4, 5, 6, 7, 8] -> [2, 3, 4, 5, 6, 8]
AC-3: Q8 <- Q1 : [1, 2, 3, 4, 5, 6, 7, 8] -> [2, 3, 4, 5, 6, 7]

MRV selected variable: Q2
Domain: [3, 4, 5, 6, 7, 8]
LCV order for Q2: [4, 6, 8, 3, 5, 7]
Trying Q2 = 4
AC-3: Q3 <- Q2 : [2, 4, 5, 6, 7, 8] -> [2, 6, 7, 8]
AC-3: Q4 <- Q2 : [2, 3, 5, 6, 7, 8] -> [3, 5, 7, 8]
AC-3: Q5 <- Q2 : [2, 3, 4, 6, 7, 8] -> [2, 3, 6, 8]
AC-3: Q6 <- Q2 : [2, 3, 4, 5, 7, 8] -> [2, 3, 5, 7]
AC-3: Q7 <- Q2 : [2, 3, 4, 5, 6, 8] -> [2, 3, 5, 6, 8]
AC-3: Q8 <- Q2 : [2, 3, 4, 5, 6, 7] -> [2, 3, 5, 6, 7]

MRV selecte

## Comparison of the Three Methods

Search step is counted whenever a candidate row value is attempted.

The three methods are compared using the same counting method.

In [53]:
print("=" * 70)
print("COMPARISON TABLE")
print("=" * 70)

print(
    "{:<35} {:<15} {:<15}".format(
        "Method",
        "Solution Found",
        "Search Steps"
    )
)

print("-" * 70)

print(
    "{:<35} {:<15} {:<15}".format(
        "Plain Backtracking",
        "Yes",
        steps1
    )
)

print(
    "{:<35} {:<15} {:<15}".format(
        "Backtracking + MRV + LCV",
        "Yes",
        steps2
    )
)

print(
    "{:<35} {:<15} {:<15}".format(
        "Backtracking + MRV + LCV + AC-3",
        "Yes",
        steps3
    )
)

print("=" * 70)

COMPARISON TABLE
Method                              Solution Found  Search Steps   
----------------------------------------------------------------------
Plain Backtracking                  Yes             876            
Backtracking + MRV + LCV            Yes             39             
Backtracking + MRV + LCV + AC-3     Yes             14             


## Conclusion

Plain backtracking tries the values in the given order and backtracks
when a conflict is found.

MRV reduces the search by selecting the variable with the fewest legal
values, while LCV tries the value that restricts the remaining variables
the least.

AC-3 performs constraint propagation by reducing domains and detecting
some failures earlier.

Therefore, the number of attempted values is reduced as we add MRV,
LCV and AC-3.